# State-Head Stable Rebound Label Screen v1

## Governance

- Baselines: canonical retention gate `model_retention_walkforward_gate_v1`; direct comparison `state_head_fold_consensus_target_v1`.
- Role: diagnose.
- Target window: fold-native teacher train rows for `wf_2022_train` and `wf_2023_2024_train`.
- Changed layer: model-selection target label discovery; no actor training or promotion.
- Hypothesis: existing observable state columns can form a duplicate-date-stable rebound label that preserves short-horizon forward-return lift and target-lift evidence after the previous fold-consensus target removed all state-head lifts.
- Acceptance criteria: at least one candidate passes duplicate-date conflict rate 0, both-fold coverage, minimum positive rows, positive 5-day forward-return lift, and positive current target-lift evidence.
- Expected trade-off: a passing label may be sparse, but it must not rely on fold-specific disagreement or future eval labels.
- Failure mode: candidates show short-term forward lift but no current target-lift support, or break across duplicate dates/folds.
- Historical success case: `rebound_state_head_lift_guard_v1` found a guarded pre-PPO recipe.
- Historical failure case: `state_head_fold_consensus_target_v1` produced zero non-trivial consensus target rows.
- Non-duplicate rationale: this screen uses the generic `stable_label_discovery` scoring layer on feature-derived candidate masks instead of reusing the prior state-head predicted labels.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "soxl_quant").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.stable_label_discovery import (
    BinaryLabelDiscoveryConfig,
    score_binary_label_candidates,
)

EXPERIMENT = "state_head_stable_rebound_label_screen_v1"
CARD_DIR = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE_TARGETS = ROOT / "runtime" / "model_selection" / "card" / "state_objective_fold_native_teacher_router_v1" / "fold_native_teacher_targets.csv"
SOURCE_CONSENSUS = ROOT / "runtime" / "model_selection" / "card" / "state_head_fold_consensus_target_v1" / "summary.json"
CARD_DIR.mkdir(parents=True, exist_ok=True)

targets = pd.read_csv(SOURCE_TARGETS, parse_dates=["datetime"]).sort_values(["fold", "datetime"]).reset_index(drop=True)
for horizon in (5, 10, 21):
    targets[f"forward_return_{horizon}"] = targets.groupby("fold")["price"].transform(lambda s: s.shift(-horizon) / s - 1.0)
targets["target_lift"] = (targets["rebound_target_position"] - targets["teacher_position"]).clip(lower=0.0)

candidate_masks: dict[str, pd.Series] = {}
for mom5 in (0.05, 0.08, 0.10, 0.12, 0.15, 0.18):
    for mom21 in (0.10, 0.20, 0.30, 0.40, 0.50):
        candidate_masks[f"mom5_{mom5:.2f}_mom21_{mom21:.2f}"] = (targets["mom_5"] >= mom5) & (targets["mom_21"] >= mom21)
for dd_low, dd_high in ((-0.25, -0.02), (-0.20, -0.02), (-0.15, -0.02), (-0.10, -0.01), (-0.25, 0.00), (-0.20, 0.00)):
    for mom5 in (0.05, 0.08, 0.10, 0.12):
        candidate_masks[f"postdd_{dd_low:.2f}_{dd_high:.2f}_mom5_{mom5:.2f}"] = targets["drawdown_63"].between(dd_low, dd_high) & (targets["mom_5"] >= mom5)
for rel in (0.05, 0.10, 0.15, 0.20, 0.25, 0.30):
    for soxx in (0.05, 0.10, 0.15, 0.20):
        candidate_masks[f"relstrength_{rel:.2f}_soxx_{soxx:.2f}"] = (targets["soxl_soxx_ratio_widen_21"] >= rel) & (targets["soxx_mom_21"] >= soxx)
for rv in (0.8, 1.0, 1.2, 1.4):
    for mom21 in (0.20, 0.30, 0.40, 0.50):
        candidate_masks[f"highrv_{rv:.1f}_mom21_{mom21:.2f}"] = (targets["rv_21"] >= rv) & (targets["mom_21"] >= mom21)
candidate_masks["source_is_rebound_cluster"] = targets["is_rebound_cluster"].astype(bool)
candidate_masks["source_rebound_state"] = targets["rebound_state"].astype(bool)

config = BinaryLabelDiscoveryConfig(
    min_positive_rows=6,
    min_positive_rate=0.002,
    max_positive_rate=0.20,
    min_fold_coverage=2,
    min_fold_positive_rows=2,
    min_forward_return_lift=0.03,
    min_target_lift=0.02,
    max_label_conflict_rate=0.0,
)
result = score_binary_label_candidates(
    targets,
    candidate_masks,
    group_col="datetime",
    fold_col="fold",
    forward_return_cols=("forward_return_5", "forward_return_10", "forward_return_21"),
    target_lift_col="target_lift",
    config=config,
)

candidate_scores = result.candidate_scores
passing = candidate_scores[candidate_scores["label_gate"].astype(bool)].copy()
selected = candidate_scores.iloc[0].to_dict()
selected_candidate = str(selected.get("candidate", ""))
selected_mask = candidate_masks[selected_candidate]
selected_rows = targets.loc[selected_mask].copy()
selected_rows.insert(1, "candidate_label", selected_candidate)

candidate_scores.to_csv(CARD_DIR / "candidate_scores.csv", index=False)
result.fold_metrics.to_csv(CARD_DIR / "fold_metrics.csv", index=False)
selected_rows.to_csv(CARD_DIR / "selected_label_rows.csv", index=False)

training_allowed = bool(len(passing) > 0)
decision = (
    "stable_rebound_label_candidate_ready_for_target_screen"
    if training_allowed
    else "stable_rebound_label_screen_no_candidate_no_training"
)
next_mechanism = (
    "build_candidate_rebound_target_contract_then_run_foldlocal_bc_checkpoint_screen"
    if training_allowed
    else "mine_new_rebound_state_factor_before_state_head_training"
)
selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "role": "diagnose",
    "changed_layer": ["model_selection", "training_target_composition", "feature"],
    "source_experiment": "state_head_fold_consensus_target_v1",
    "direct_comparison_baseline": "state_head_fold_consensus_target_v1",
    "decision": decision,
    "promotion_allowed": False,
    "training_allowed": training_allowed,
    "target_max_dd_pct": 20.0,
    "target_return_floor_pct": 213.143462,
    "target_reference_return_pct": 132.077978,
    "candidate_count": int(len(candidate_scores)),
    "passing_candidate_count": int(len(passing)),
    "selected_candidate": selected_candidate,
    "selected_label_gate": bool(selected.get("label_gate", False)),
    "selected_positive_rows": int(selected.get("positive_rows", 0)),
    "selected_positive_groups": int(selected.get("positive_groups", 0)),
    "selected_primary_forward_return_lift": float(selected.get("primary_forward_return_lift", np.nan)),
    "selected_forward_return_10_lift": float(selected.get("forward_return_10_lift", np.nan)),
    "selected_forward_return_21_lift": float(selected.get("forward_return_21_lift", np.nan)),
    "selected_target_lift_mean": float(selected.get("target_lift_mean", np.nan)),
    "selected_label_conflict_rate": float(selected.get("label_conflict_rate", np.nan)),
    "selected_covered_folds": int(selected.get("covered_folds", 0)),
    "config": config.__dict__,
    "source_targets": str(SOURCE_TARGETS.relative_to(ROOT)),
    "source_consensus": str(SOURCE_CONSENSUS.relative_to(ROOT)),
    "sidecars": [
        "candidate_scores.csv",
        "fold_metrics.csv",
        "selected_label_rows.csv",
        "selection.json",
        "summary.json",
    ],
    "next_mechanism": next_mechanism,
}
conclusion = [
    f"Screened {len(candidate_scores)} feature-derived candidate labels against duplicate-date stability and two-fold coverage.",
    f"Passing candidates: {len(passing)}.",
    f"Best candidate is {selected_candidate}: 5d forward lift {selection['selected_primary_forward_return_lift']:.4f}, 10d lift {selection['selected_forward_return_10_lift']:.4f}, 21d lift {selection['selected_forward_return_21_lift']:.4f}, target-lift mean {selection['selected_target_lift_mean']:.4f}.",
    "Do not start state-head training from this label screen unless a later factor screen supplies a passing stable label with positive target-lift evidence." if not training_allowed else "A candidate passed the label screen; build a target contract before any PPO.",
]
summary = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "promotion_allowed": False,
    "training_allowed": training_allowed,
    "direct_comparison_baseline": "state_head_fold_consensus_target_v1",
    "source_experiment": "state_head_fold_consensus_target_v1",
    "next_mechanism": next_mechanism,
    "selection": selection,
    "conclusion": conclusion,
}
(CARD_DIR / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print(json.dumps({
    "decision": decision,
    "training_allowed": training_allowed,
    "selected_candidate": selected_candidate,
    "passing_candidate_count": int(len(passing)),
    "selected_primary_forward_return_lift": selection["selected_primary_forward_return_lift"],
    "selected_target_lift_mean": selection["selected_target_lift_mean"],
    "next_mechanism": next_mechanism,
}, indent=2, sort_keys=True))
candidate_scores.head(12)

{
  "decision": "stable_rebound_label_screen_no_candidate_no_training",
  "next_mechanism": "mine_new_rebound_state_factor_before_state_head_training",
  "passing_candidate_count": 0,
  "selected_candidate": "highrv_1.4_mom21_0.50",
  "selected_primary_forward_return_lift": 0.04295490619199901,
  "selected_target_lift_mean": 0.0,
  "training_allowed": false
}


,candidate,rows,groups,positive_rows,positive_groups,positive_rate,label_conflict_groups,label_conflict_rate,forward_return_5_positive_mean,forward_return_5_negative_mean,...,target_lift_mean,target_lift_median,folds,covered_folds,min_fold_positive_rows,forward_gate,target_gate,fold_gate,label_gate,label_loss
0,highrv_1.4_mom21_0.50,1261,756,16,12,0.012688,0,0.0,0.055971,0.013016,...,0.000000,0.0,2,2,4,True,False,True,False,0.020000
1,relstrength_0.20_soxx_0.20,1261,756,22,16,0.017446,0,0.0,0.041819,0.013060,...,0.000000,0.0,2,2,6,False,False,True,False,0.024963
2,relstrength_0.25_soxx_0.20,1261,756,22,16,0.017446,0,0.0,0.041819,0.013060,...,0.000000,0.0,2,2,6,False,False,True,False,0.024963
3,relstrength_0.30_soxx_0.20,1261,756,22,16,0.017446,0,0.0,0.041819,0.013060,...,0.000000,0.0,2,2,6,False,False,True,False,0.024963
4,relstrength_0.05_soxx_0.20,1261,756,24,17,0.019033,0,0.0,0.041164,0.013026,...,0.000000,0.0,2,2,7,False,False,True,False,0.027448
5,relstrength_0.10_soxx_0.20,1261,756,24,17,0.019033,0,0.0,0.041164,0.013026,...,0.000000,0.0,2,2,7,False,False,True,False,0.027448
6,relstrength_0.15_soxx_0.20,1261,756,24,17,0.019033,0,0.0,0.041164,0.013026,...,0.000000,0.0,2,2,7,False,False,True,False,0.027448
7,postdd_-0.20_-0.02_mom5_0.08,1261,756,115,60,0.091197,0,0.0,0.031407,0.011794,...,0.003840,0.0,2,2,55,False,False,True,False,0.057709
8,postdd_-0.15_-0.02_mom5_0.12,1261,756,50,25,0.039651,0,0.0,0.031011,0.012854,...,0.008831,0.0,2,2,25,False,False,True,False,0.058543
9,highrv_1.4_mom21_0.30,1261,756,42,29,0.033307,0,0.0,0.031683,0.012936,...,0.000000,0.0,2,2,13,False,False,True,False,0.065012


## Conclusion

This notebook is the single-run conclusion for the stable rebound label screen. The current feature-derived screen does not authorize state-head training unless `summary.json` reports `training_allowed=true`. A `training_allowed=false` result means the previous `state_head_fold_consensus_target_v1` blocker is real and the next useful work is mining a new rebound state factor before any BC/PPO loop.